In [ ]:
# --- Deducción Lagrangiana: Péndulo Doble ---
import sympy as sp
from sympy.physics.mechanics import dynamicsymbols
from IPython.display import display, Markdown

display(Markdown("### Deducción de Ecuaciones: Péndulo Doble"))

t = sp.Symbol('t')
m1, m2, l1, l2, g = sp.symbols('m1 m2 l1 l2 g')
theta1, theta2 = dynamicsymbols('theta1 theta2')
theta1_d, theta2_d = dynamicsymbols('theta1 theta2', 1)

# Posiciones Cartesianas
x1 = l1 * sp.sin(theta1)
y1 = -l1 * sp.cos(theta1)
x2 = x1 + l2 * sp.sin(theta2)
y2 = y1 - l2 * sp.cos(theta2)

# Velocidades al cuadrado
v1_2 = sp.diff(x1, t)**2 + sp.diff(y1, t)**2
v2_2 = sp.diff(x2, t)**2 + sp.diff(y2, t)**2

# Energía Cinética (T) y Potencial (V)
T = sp.Rational(1, 2) * m1 * v1_2 + sp.Rational(1, 2) * m2 * v2_2
V = m1 * g * y1 + m2 * g * y2

# Lagrangiano
L_lag = sp.simplify(T - V)

# Ecuaciones de Euler-Lagrange
eq1 = sp.simplify(sp.diff(sp.diff(L_lag, theta1_d), t) - sp.diff(L_lag, theta1))
eq2 = sp.simplify(sp.diff(sp.diff(L_lag, theta2_d), t) - sp.diff(L_lag, theta2))

display(Markdown("**Lagrangiano ($L$):**"))
display(sp.Eq(sp.Symbol('L'), L_lag))
display(Markdown("**Ecuación de Euler-Lagrange respecto a $\\theta_1$:**"))
display(sp.Eq(eq1, 0))
display(Markdown("**Ecuación de Euler-Lagrange respecto a $\\theta_2$:**"))
display(sp.Eq(eq2, 0))

In [2]:
import numpy as np
from scipy.integrate import solve_ivp
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML, display
from ipywidgets import interact_manual, FloatSlider

# Ecuaciones del movimiento para el péndulo doble
def double_pendulum_eqs(t, y, l1, l2, m1, m2, g):
    theta1, z1, theta2, z2 = y
    delta = theta2 - theta1

    den1 = (m1 + m2)*l1 - m2*l1*np.cos(delta)*np.cos(delta)
    den2 = (l2/l1)*den1

    dydt = np.zeros_like(y)
    dydt[0] = z1
    dydt[1] = (m2*l1*z1*z1*np.sin(delta)*np.cos(delta)
             + m2*g*np.sin(theta2)*np.cos(delta)
             + m2*l2*z2*z2*np.sin(delta)
             - (m1+m2)*g*np.sin(theta1)) / den1
    dydt[2] = z2
    dydt[3] = (-m2*l2*z2*z2*np.sin(delta)*np.cos(delta)
             + (m1+m2)*(g*np.sin(theta1)*np.cos(delta) 
             - l1*z1*z1*np.sin(delta) - g*np.sin(theta2))) / den2
    return dydt

# Integrar y animar el péndulo doble
def simulate_and_animate(l1, l2, m1, m2, g):
    y0 = [np.pi/2, 0, np.pi/2, 0]  # condiciones iniciales
    t_span = (0, 10) # 10 segundos de simulación para generar más rápido
    t_eval = np.linspace(0, 10, 300)
    sol = solve_ivp(double_pendulum_eqs, t_span, y0, t_eval=t_eval, args=(l1, l2, m1, m2, g), method='RK45')

    theta1 = sol.y[0]
    theta2 = sol.y[2]

    x1 = l1 * np.sin(theta1)
    y1 = -l1 * np.cos(theta1)
    x2 = x1 + l2 * np.sin(theta2)
    y2 = y1 - l2 * np.cos(theta2)

    fig, ax = plt.subplots(figsize=(5,5))
    ax.set_aspect('equal')
    
    # Ajustar ejes dinámicamente según las longitudes
    limit = l1 + l2 + 0.5
    ax.set_xlim(-limit, limit)
    ax.set_ylim(-limit, limit)
    
    line, = ax.plot([], [], 'o-', lw=2)
    plt.close(fig) # Cerrar la figura estática inicial

    def init():
        line.set_data([], [])
        return line,

    def update(i):
        thisx = [0, x1[i], x2[i]]
        thisy = [0, y1[i], y2[i]]
        line.set_data(thisx, thisy)
        return line,

    ani = FuncAnimation(fig, update, frames=len(t_eval),
                        init_func=init, blit=True, interval=33)
    
    # Renderizar la animación en HTML5
    display(HTML(ani.to_jshtml()))

# Crear interfaz interactiva para Jupyter Notebooks
_ = interact_manual(
    simulate_and_animate,
    l1=FloatSlider(value=1.0, min=0.2, max=2.0, step=0.1, description='Longitud 1'),
    l2=FloatSlider(value=1.0, min=0.2, max=2.0, step=0.1, description='Longitud 2'),
    m1=FloatSlider(value=1.0, min=0.1, max=3.0, step=0.1, description='Masa 1'),
    m2=FloatSlider(value=1.0, min=0.1, max=3.0, step=0.1, description='Masa 2'),
    g=FloatSlider(value=9.81, min=0.0, max=20.0, step=0.1, description='Gravedad')
)

interactive(children=(FloatSlider(value=1.0, description='Longitud 1', max=2.0, min=0.2), FloatSlider(value=1.…

In [ ]:
# --- Deducción Lagrangiana: Péndulo Elástico (con resorte) ---
import sympy as sp
from sympy.physics.mechanics import dynamicsymbols
from IPython.display import display, Markdown

display(Markdown("### Deducción de Ecuaciones: Péndulo Elástico (Resorte)"))

t = sp.Symbol('t')
m, k, L0, g = sp.symbols('m k L0 g')
r, theta = dynamicsymbols('r theta')
r_d, theta_d = dynamicsymbols('r theta', 1)

# Posiciones Cartesianas
x = r * sp.sin(theta)
y = -r * sp.cos(theta)

# Velocidad al cuadrado
v_2 = sp.simplify(sp.diff(x, t)**2 + sp.diff(y, t)**2)

# Energía Cinética (T) y Potencial (V general: gravitatorio + elástico)
T = sp.Rational(1, 2) * m * v_2
V = m * g * y + sp.Rational(1, 2) * k * (r - L0)**2

# Lagrangiano
L_lag = T - V

# Ecuaciones de Euler-Lagrange
eq_r = sp.simplify(sp.diff(sp.diff(L_lag, r_d), t) - sp.diff(L_lag, r))
eq_theta = sp.simplify(sp.diff(sp.diff(L_lag, theta_d), t) - sp.diff(L_lag, theta))

display(Markdown("**Lagrangiano ($L$):**"))
display(sp.Eq(sp.Symbol('L'), L_lag))
display(Markdown("**Ecuación radial ($r$):**"))
display(sp.Eq(eq_r, 0))
display(Markdown("**Ecuación angular ($\\theta$):**"))
display(sp.Eq(eq_theta, 0))

In [4]:
import numpy as np
from scipy.integrate import solve_ivp
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML, display
from ipywidgets import interact_manual, FloatSlider

# Ecuaciones del movimiento para un péndulo elástico (con resorte)
def spring_pendulum_eqs(t, y, m, k, L0, g):
    # y = [r, dr/dt, theta, dtheta/dt]
    r, z_r, theta, z_theta = y
    
    drdt = z_r
    # Ecuación radial: m*r'' = m*r*theta'^2 + m*g*cos(theta) - k*(r - L0)
    dz_rdt = r * z_theta**2 + g * np.cos(theta) - (k / m) * (r - L0)
    
    dthetadt = z_theta
    # Ecuación angular: r*theta'' + 2*r'*theta' + g*sin(theta) = 0
    dz_thetadt = - (2.0 * z_r * z_theta) / r - (g / r) * np.sin(theta)
    
    return [drdt, dz_rdt, dthetadt, dz_thetadt]

# Función para generar la forma del resorte de manera zig-zag
def generate_spring_coords(x0, y0, x1, y1, width=0.1, n_coils=10):
    dx = x1 - x0
    dy = y1 - y0
    length = np.sqrt(dx**2 + dy**2)
    
    # Vector direccional del resorte
    dir_x = dx / length
    dir_y = dy / length
    
    # Vector normal para dibujar las espiras del resorte
    norm_x = -dir_y * width
    norm_y = dir_x * width
    
    # Construir puntos del resorte
    points_x, points_y = [x0], [y0]
    
    for i in range(1, n_coils * 2):
        frac = i / (n_coils * 2)
        px = x0 + dir_x * length * frac
        py = y0 + dir_y * length * frac
        
        # Alternar hacia un lado normal y luego al otro
        if i % 2 == 1:
            px += norm_x
            py += norm_y
        else:
            px -= norm_x
            py -= norm_y
            
        points_x.append(px)
        points_y.append(py)
        
    points_x.append(x1)
    points_y.append(y1)
    
    return points_x, points_y

def simulate_and_animate_spring(m, k, L0, g, r0_factor, theta0_deg, v_r0, v_theta0):
    theta0 = np.radians(theta0_deg)
    r0 = L0 * r0_factor # Longitud inicial como un factor de la longitud natural
    y0 = [r0, v_r0, theta0, v_theta0]  # [r, dr/dt, theta, dtheta/dt] iniciales
    
    t_span = (0, 15) # 15 segundos
    t_eval = np.linspace(0, 15, 450)
    sol = solve_ivp(spring_pendulum_eqs, t_span, y0, t_eval=t_eval, args=(m, k, L0, g), method='RK45')

    r_out = sol.y[0]
    theta_out = sol.y[2]

    # Convertir coordenadas polares a cartesianas
    x = r_out * np.sin(theta_out)
    y = -r_out * np.cos(theta_out)

    fig, ax = plt.subplots(figsize=(6, 6))
    ax.set_aspect('equal')
    
    # Ajustar ventana
    max_r = max(np.max(r_out), L0) * 1.1
    ax.set_xlim(-max_r, max_r)
    ax.set_ylim(-max_r, max_r * 0.2)
    
    # Dibujar el pivote
    ax.plot(0, 0, 'ks', markersize=8)
    
    # Elementos animados: ahora el resorte es una línea discontinua ancha
    spring_line, = ax.plot([], [], 'limegreen', lw=1.5, label='Resorte')      
    mass, = ax.plot([], [], 'ro', markersize=15, label='Masa', zorder=5)
    trail, = ax.plot([], [], 'b--', alpha=0.5, lw=1) # Rastro de la trayectoria

    plt.close(fig) # Cierra figura inicial para evitar duplicados

    def init():
        spring_line.set_data([], [])
        mass.set_data([], [])
        trail.set_data([], [])
        return spring_line, mass, trail

    def update(i):
        # Dibujar forma de resorte zig-zag en lugar de una línea simple
        sx, sy = generate_spring_coords(0, 0, x[i], y[i], width=max_r*0.03, n_coils=12)
        spring_line.set_data(sx, sy)
        
        mass.set_data([x[i]], [y[i]])
        
        # Dibujar rastro de los últimos 60 frames
        start_idx = max(0, i - 60)
        trail.set_data(x[start_idx:i], y[start_idx:i])
        return spring_line, mass, trail

    ani = FuncAnimation(fig, update, frames=len(t_eval),
                        init_func=init, blit=True, interval=33)
    
    display(HTML(ani.to_jshtml()))

# Interfaz interactiva 
_ = interact_manual(
    simulate_and_animate_spring,
    m=FloatSlider(value=1.0, min=0.1, max=5.0, step=0.1, description='Masa (kg)'),
    k=FloatSlider(value=20.0, min=1.0, max=100.0, step=1.0, description='Const. Resorte (N/m)'),
    L0=FloatSlider(value=1.5, min=0.5, max=5.0, step=0.1, description='Long. Natural L0(m)'),
    g=FloatSlider(value=9.81, min=0.0, max=20.0, step=0.1, description='Gravedad (m/s²)'),
    r0_factor=FloatSlider(value=1.3, min=0.5, max=2.0, step=0.1, description='Estiramiento Inicial (x L0)'),
    theta0_deg=FloatSlider(value=45.0, min=-90.0, max=90.0, step=1.0, description='Ángulo Inicial (deg)'),
    v_r0=FloatSlider(value=0.0, min=-5.0, max=5.0, step=0.1, description='Vel. Radial (m/s)'),
    v_theta0=FloatSlider(value=0.0, min=-5.0, max=5.0, step=0.1, description='Vel. Angular (rad/s)')
)

interactive(children=(FloatSlider(value=1.0, description='Masa (kg)', max=5.0, min=0.1), FloatSlider(value=20.…

In [5]:
import numpy as np
from scipy.integrate import solve_ivp
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML, display
from ipywidgets import interact_manual, FloatSlider

# Ecuaciones del movimiento para el péndulo doble
def double_pendulum_eqs(t, y, l1, l2, m1, m2, g):
    theta1, z1, theta2, z2 = y
    delta = theta2 - theta1

    den1 = (m1 + m2)*l1 - m2*l1*np.cos(delta)*np.cos(delta)
    den2 = (l2/l1)*den1

    dydt = np.zeros_like(y)
    dydt[0] = z1
    dydt[1] = (m2*l1*z1*z1*np.sin(delta)*np.cos(delta)
             + m2*g*np.sin(theta2)*np.cos(delta)
             + m2*l2*z2*z2*np.sin(delta)
             - (m1+m2)*g*np.sin(theta1)) / den1
    dydt[2] = z2
    dydt[3] = (-m2*l2*z2*z2*np.sin(delta)*np.cos(delta)
             + (m1+m2)*(g*np.sin(theta1)*np.cos(delta) 
             - l1*z1*z1*np.sin(delta) - g*np.sin(theta2))) / den2
    return dydt

# Integrar y animar el péndulo doble mostrando los ángulos
def simulate_and_animate_angles(l1, l2, m1, m2, g):
    y0 = [np.pi/2, 0, np.pi/2, 0]  # condiciones iniciales (90 grados)
    t_span = (0, 10) 
    t_eval = np.linspace(0, 10, 300)
    sol = solve_ivp(double_pendulum_eqs, t_span, y0, t_eval=t_eval, args=(l1, l2, m1, m2, g), method='RK45')

    theta1 = sol.y[0]
    theta2 = sol.y[2]

    x1 = l1 * np.sin(theta1)
    y1 = -l1 * np.cos(theta1)
    x2 = x1 + l2 * np.sin(theta2)
    y2 = y1 - l2 * np.cos(theta2)

    fig, ax = plt.subplots(figsize=(6, 6))
    ax.set_aspect('equal')
    
    limit = l1 + l2 + 0.5
    ax.set_xlim(-limit, limit)
    ax.set_ylim(-limit, limit*0.4)
    
    # Eje vertical de referencia para el primer péndulo
    ax.plot([0, 0], [0, -limit], 'k--', alpha=0.3)
    
    # Elementos dinámicos
    line, = ax.plot([], [], 'o-', lw=2, markersize=8)
    vert_line2, = ax.plot([], [], 'k--', alpha=0.3)  # Eje vertical de referencia para el segundo péndulo
    
    # Textos que muestran los valores numéricos de los ángulos
    # Se colocarán en la esquina superior izquierda
    text_theta1 = ax.text(-limit*0.9, limit*0.2, '', fontsize=12, color='tab:blue', fontweight='bold')
    text_theta2 = ax.text(-limit*0.9, limit*0.05, '', fontsize=12, color='tab:orange', fontweight='bold')

    plt.close(fig) # Cierra la figura inicial

    def init():
        line.set_data([], [])
        vert_line2.set_data([], [])
        text_theta1.set_text('')
        text_theta2.set_text('')
        return line, vert_line2, text_theta1, text_theta2

    def update(i):
        thisx = [0, x1[i], x2[i]]
        thisy = [0, y1[i], y2[i]]
        line.set_data(thisx, thisy)
        
        # Actualizamos la línea vertical de referencia que cuelga desde la masa 1
        vert_line2.set_data([x1[i], x1[i]], [y1[i], y1[i]-l2])
        
        # Convertimos los ángulos a grados y los normalizamos a un rango de [-180, 180] 
        # para que sean más fáciles de leer
        deg1 = np.degrees(theta1[i]) % 360
        if deg1 > 180: deg1 -= 360
            
        deg2 = np.degrees(theta2[i]) % 360
        if deg2 > 180: deg2 -= 360
            
        text_theta1.set_text(f'$\\theta_1$: {deg1:6.1f}°')
        text_theta2.set_text(f'$\\theta_2$: {deg2:6.1f}°')
        
        return line, vert_line2, text_theta1, text_theta2

    ani = FuncAnimation(fig, update, frames=len(t_eval),
                        init_func=init, blit=True, interval=33)
    
    display(HTML(ani.to_jshtml()))

# Interfaz interactiva 
_ = interact_manual(
    simulate_and_animate_angles,
    l1=FloatSlider(value=1.0, min=0.2, max=2.0, step=0.1, description='Longitud 1'),
    l2=FloatSlider(value=1.0, min=0.2, max=2.0, step=0.1, description='Longitud 2'),
    m1=FloatSlider(value=1.0, min=0.1, max=3.0, step=0.1, description='Masa 1'),
    m2=FloatSlider(value=1.0, min=0.1, max=3.0, step=0.1, description='Masa 2'),
    g=FloatSlider(value=9.81, min=0.0, max=20.0, step=0.1, description='Gravedad')
)

interactive(children=(FloatSlider(value=1.0, description='Longitud 1', max=2.0, min=0.2), FloatSlider(value=1.…

In [ ]:
# --- Deducción Lagrangiana: Péndulo Cónico / Esférico ---
import sympy as sp
from sympy.physics.mechanics import dynamicsymbols
from IPython.display import display, Markdown

display(Markdown("### Deducción de Ecuaciones: Péndulo Cónico (Esférico 3D)"))

t = sp.Symbol('t')
m, L, g = sp.symbols('m L g')
theta, phi = dynamicsymbols('theta phi') # Polar y azimutal
theta_d, phi_d = dynamicsymbols('theta phi', 1)

# Coordenadas Cartesianas 3D
x = L * sp.sin(theta) * sp.cos(phi)
y = L * sp.sin(theta) * sp.sin(phi)
z = -L * sp.cos(theta)

# Velocidad al cuadrado
v_2 = sp.simplify(sp.diff(x, t)**2 + sp.diff(y, t)**2 + sp.diff(z, t)**2)

# Energías
T = sp.Rational(1, 2) * m * v_2
V = m * g * z

# Lagrangiano
L_lag = T - V

# Ecuaciones de Euler-Lagrange
eq_theta = sp.simplify(sp.diff(sp.diff(L_lag, theta_d), t) - sp.diff(L_lag, theta))
eq_phi = sp.simplify(sp.diff(sp.diff(L_lag, phi_d), t) - sp.diff(L_lag, phi))

display(Markdown("**Lagrangiano ($L$):**"))
display(sp.Eq(sp.Symbol('L'), L_lag))
display(Markdown("**Ecuación de Euler-Lagrange del ángulo Polar ($\\theta$):**"))
display(sp.Eq(eq_theta, 0))
display(Markdown("**Ecuación de Euler-Lagrange del ángulo Azimutal ($\\phi$):**"))
display(sp.Eq(eq_phi, 0))

In [6]:
import numpy as np
from scipy.integrate import solve_ivp
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from matplotlib.animation import FuncAnimation
from IPython.display import HTML, display
from ipywidgets import interact_manual, FloatSlider

# Ecuaciones del movimiento para un péndulo esférico/cónico
def spherical_pendulum_eqs(t, y, L, g):
    # y = [theta, dtheta/dt, phi, dphi/dt]
    theta, z_theta, phi, z_phi = y
    
    # Evitar singularidades numéricas exactamente en 0 o pi
    if abs(theta) < 1e-7:
        theta = 1e-7
        
    dtheta_dt = z_theta
    # Aceleración polar (theta)
    dz_theta_dt = np.sin(theta) * np.cos(theta) * z_phi**2 - (g / L) * np.sin(theta)
    
    dphi_dt = z_phi
    # Aceleración azimutal (phi), conservando el momento angular
    dz_phi_dt = -2.0 * z_theta * z_phi / np.tan(theta)
    
    return [dtheta_dt, dz_theta_dt, dphi_dt, dz_phi_dt]

def simulate_conical_pendulum(L, m, g, theta0_deg, omega_factor):
    theta0 = np.radians(theta0_deg)
    
    # Para que un péndulo sea puramente "cónico" (orbite en un círculo perfecto 
    # sin oscilar arriba y abajo), necesita una velocidad angular inicial exacta:
    # w = sqrt(g / (L * cos(theta_0)))
    if np.cos(theta0) > 0:
        w_conical = np.sqrt(g / (L * np.cos(theta0)))
    else:
        w_conical = 0
        
    # Inicializamos la simulación. 
    # omega_factor = 1 genera un péndulo cónico perfecto.
    # != 1 genera un péndulo esférico general con figuras más complejas.
    z_phi0 = w_conical * omega_factor
    
    y0 = [theta0, 0.0, 0.0, z_phi0]
    
    t_span = (0, 10)
    t_eval = np.linspace(0, 10, 300)
    sol = solve_ivp(spherical_pendulum_eqs, t_span, y0, t_eval=t_eval, args=(L, g), method='RK45')
    
    theta = sol.y[0]
    phi = sol.y[2]
    
    # Transformación de coordenadas cartesianas (3D)
    x = L * np.sin(theta) * np.cos(phi)
    y = L * np.sin(theta) * np.sin(phi)
    z = -L * np.cos(theta)
    
    fig = plt.figure(figsize=(7, 7))
    ax = fig.add_subplot(111, projection='3d')
    
    # Ajustes del entorno 3D
    limit = L * 1.1
    ax.set_xlim([-limit, limit])
    ax.set_ylim([-limit, limit])
    ax.set_zlim([-limit, 0.1])
    ax.set_xlabel('Eje X')
    ax.set_ylabel('Eje Y')
    ax.set_zlabel('Eje Z (vertical)')
    
    # Pivote
    ax.plot([0], [0], [0], 'ks', markersize=8)
    
    # Elementos dinámicos en 3D
    line, = ax.plot([], [], [], 'k-', lw=2, label='Cuerda')
    mass, = ax.plot([], [], [], 'ro', markersize=15, label='Masa')
    trail, = ax.plot([], [], [], 'b--', alpha=0.5, label='Trayectoria')

    plt.close(fig)

    def init():
        line.set_data([], [])
        line.set_3d_properties([])
        mass.set_data([], [])
        mass.set_3d_properties([])
        trail.set_data([], [])
        trail.set_3d_properties([])
        return line, mass, trail

    def update(i):
        # Actualizando datos en Matplotlib 3D
        line.set_data([0, x[i]], [0, y[i]])
        line.set_3d_properties([0, z[i]])
        
        mass.set_data([x[i]], [y[i]])
        mass.set_3d_properties([z[i]])
        
        start_idx = max(0, i - 120)  # Larga cola para ver el camino circular
        trail.set_data(x[start_idx:i], y[start_idx:i])
        trail.set_3d_properties(z[start_idx:i])
        
        return line, mass, trail

    # Animación
    ani = FuncAnimation(fig, update, frames=len(t_eval),
                        init_func=init, blit=False, interval=33)
    
    display(HTML(ani.to_jshtml()))

# Interfaz interactiva 
_ = interact_manual(
    simulate_conical_pendulum,
    L=FloatSlider(value=2.0, min=0.5, max=5.0, step=0.1, description='Longitud (m)'),
    m=FloatSlider(value=1.0, min=0.1, max=5.0, step=0.1, description='Masa (kg)'),
    g=FloatSlider(value=9.81, min=0.0, max=20.0, step=0.1, description='Gravedad (m/s²)'),
    theta0_deg=FloatSlider(value=45.0, min=5.0, max=85.0, step=1.0, description='Ángulo (deg)'),
    omega_factor=FloatSlider(value=1.0, min=0.0, max=2.0, step=0.1, description='Factor de Impulso')
)

interactive(children=(FloatSlider(value=2.0, description='Longitud (m)', max=5.0, min=0.5), FloatSlider(value=…